In [1]:
from pypdf import PdfReader
import pandas as pd
import chromadb
from sentence_transformers import SentenceTransformer

In [2]:
pdf_path = "Salesforce_MSA.pdf"
reader = PdfReader(pdf_path)
print("Total Pages :", len(reader.pages))

Total Pages : 17


In [3]:
import re
chunk_size = 1000
chunk_overlap = 175
page_chunks = []
for page_num, page in enumerate(reader.pages, start=1):
    page_text = page.extract_text()
    if not page_text:
        continue
# Clean page text
    
    page_text = re.sub(r"\s+", " ", page_text).strip()
    start = 0
    while start < len(page_text):
        end = start + chunk_size
        page_chunks.append({
            "text": page_text[start:end],
            "page": page_num,
            "location": f"Page {page_num}"
        })
        start = end - chunk_overlap
print("Total Chunks:", len(page_chunks))

Total Chunks: 100


In [4]:
print(page_chunks[0])

{'text': 'MAIN SERVICES AGREEMENT THIS MAIN SERVICES AGREEMENT GOVERNS CUSTOMER’S ACQUISITION AND USE OF SFDC SERVICES. CAPITALIZED TERMS HAVE THE DEFINITIONS SET FORTH HEREIN. IF CUSTOMER REGISTERS FOR A FREE TRIAL OF SFDC SERVICES OR FOR FREE SERVICES, THE APPLICABLE PROVISIONS OF THIS AGREEMENT WILL ALSO GOVERN THAT FREE TRIAL OR THOSE FREE SERVICES. BY ACCEPTING THIS AGREEMENT, BY (1) CLICKING A BOX INDICATING ACCEPTANCE, (2) EXECUTING AN ORDER FORM THAT REFERENCES THIS AGREEMENT, OR (3) USING FREE SERVICES, CUSTOMER AGREES TO THE TERMS OF THIS AGREEMENT. IF THE INDIVIDUAL ACCEPTING THIS AGREEMENT IS ACCEPTING ON BEHALF OF A COMPANY OR OTHER LEGAL ENTITY, SUCH INDIVIDUAL REPRESENTS THAT THEY HAVE THE AUTHORITY TO BIND SUCH ENTITY AND ITS AFFILIATES TO THESE TERMS AND CONDITIONS, IN WHICH CASE THE TERM “CUSTOMER” SHALL REFER TO SUCH ENTITY AND ITS AFFILIATES. IF THE INDIVIDUAL ACCEPTING THIS AGREEMENT DOES NOT HAVE SUCH AUTHORITY, OR DOES NOT AGREE WITH THESE TERMS AND CONDITIONS, S

In [5]:
chunk_texts = [
    item["text"]
    for item in page_chunks
]

print("Total Text Chunks:", len(chunk_texts))

Total Text Chunks: 100


In [6]:
print(chunk_texts[0][:200])

MAIN SERVICES AGREEMENT THIS MAIN SERVICES AGREEMENT GOVERNS CUSTOMER’S ACQUISITION AND USE OF SFDC SERVICES. CAPITALIZED TERMS HAVE THE DEFINITIONS SET FORTH HEREIN. IF CUSTOMER REGISTERS FOR A FREE 


In [7]:
chunk_lengths = [len(item["text"]) for item in page_chunks]
print("Total Chunks:", len(chunk_lengths))
print("Smallest Chunk:", min(chunk_lengths))
print("Largest Chunk:", max(chunk_lengths))
print("Average Chunk Size:", round(sum(chunk_lengths)/len(chunk_lengths)))

Total Chunks: 100
Smallest Chunk: 45
Largest Chunk: 1000
Average Chunk Size: 904


In [8]:
#embedding
from sentence_transformers import SentenceTransformer
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

In [9]:
chunk_embeddings = embedding_model.encode(
    chunk_texts,
    show_progress_bar=True
)
print(chunk_embeddings.shape)

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

(100, 384)


In [10]:
#chroma db
import chromadb
client = chromadb.Client()

In [11]:
collection = client.create_collection(
    name="salesforce_msa"
)

In [12]:
collection.add(
    embeddings=chunk_embeddings.tolist(),
    documents=chunk_texts,
    metadatas=[
        {
            "page": item["page"],
            "location": item["location"]
        }
        for item in page_chunks
    ],
    ids=[
        f"chunk_{i}"
        for i in range(len(page_chunks))
    ]
)

In [13]:
print("Vectors Stored:", collection.count())

Vectors Stored: 100


In [70]:
#retirival
def retrieve_chunks(question, collection, embedding_model, top_k=5):

    question_embedding = embedding_model.encode(question)

    results = collection.query(
        query_embeddings=[question_embedding.tolist()],
        n_results=top_k
    )

    return results

In [72]:
question = "When is an invoice due after it is issued?"

results = retrieve_chunks(
    question,
    collection,
    embedding_model
)

In [74]:
def score_signal(score):
    if score >= 0.85:
        return "Strong"
    elif score >= 0.70:
        return "Good"
    elif score >= 0.50:
        return "Weak"
    else:
        return "Poor"

In [80]:
#similarity report
similarity_report = []
for rank in range(5):
    distance = results["distances"][0][rank]
    similarity_score = round(max(0, 1 - (distance / 2)), 2)
    similarity_report.append({
        "Rank": rank + 1,
        "Location":
        results["metadatas"][0][rank]["location"],
        "Similarity Score":
        similarity_score,
        "Score Signal":
        score_signal(similarity_score),
        "Chunk Preview":
        results["documents"][0][rank][:120]

    })

In [82]:
import pandas as pd
report_df = pd.DataFrame(similarity_report)
report_df

,Rank,Location,Similarity Score,Score Signal,Chunk Preview
0,1,Page 5,0.61,Weak,orizes SFDC to charge such credit card for all...
1,2,Page 13,0.60,Weak,above limitations of liability also apply in ...
2,3,Page 14,0.59,Weak,reserves the right to provide any invoice copy...
3,4,Page 13,0.56,Weak,"ered office address, VAT number, tax/fiscal co..."
4,5,Page 15,0.55,Weak,ast the following information in writing to fa...


In [84]:
question = "What are the payment terms?"

results = retrieve_chunks(
    question,
    collection,
    embedding_model
)

for i in range(5):

    print("\nRANK:", i+1)
    print(results["metadatas"][0][i])
    print(results["documents"][0][i][:250])


RANK: 1
{'page': 15, 'location': 'Page 15'}
ot result in an extension of the payment term set out in the “Invoicing and Payment” section above, and such term shall still be calculated from the date of the original invoice. SFDC reserves the right to provide any invoice copy in electronic form 

RANK: 2
{'location': 'Page 5', 'page': 5}
or other compensation, in the event a Non-SFDC Application no longer interoperates with the corresponding Service features in a manner acceptable to SFDC or is no longer available. 5. FEES AND PAYMENT 5.1 Fees . Customer will pay all fees specified i

RANK: 3
{'page': 9, 'location': 'Page 9'}
ization, or sale of all or substantially all of its assets. Notwithstanding the foregoing, if a party is acquired by, sells substantially all of its assets to, or undergoes a change of control in favor of, a direct competitor of the other party, then

RANK: 4
{'page': 4, 'location': 'Page 4'}
IREMENTS, (B) CUSTOMER’S USE OF THE FREE SERVICES WILL BE UNINTERRUPTED, 

In [86]:
def build_context(results):
    context = ""
    for chunk in results["documents"][0]:
        context += chunk
        context += "\n\n"
    return context

In [90]:
context = build_context(results)
print(context[:2000])

ot result in an extension of the payment term set out in the “Invoicing and Payment” section above, and such term shall still be calculated from the date of the original invoice. SFDC reserves the right to provide any invoice copy in electronic form via email in addition to the electronic invoicing described herein. 12.17 Local Law Requirements: United Kingdom. With respect to Customers domiciled in the United Kingdom, Section 12.3 “Entire Agreement and Order of Precedence” of this Agreement is replaced with the following section: 12.3 Entire Agreement and Order of Precedence. This Agreement is the entire agreement between SFDC and Customer regarding Customer’s use of Services and supersedes all prior and contemporaneous agreements, proposals or representations, written or oral, concerning its subject matter. No representation, undertaking or promise shall be taken to have been given or be implied from anything said or written in negotiations between the parties prior to this Agreement

In [271]:
import os
from dotenv import load_dotenv
load_dotenv()
genai.configure(api_key=os.getenv("GEMINI_API_KEY"))
model = genai.GenerativeModel("gemini-3.1-flash-lite")


In [134]:
model = genai.GenerativeModel("gemini-flash-lite-latest")

In [136]:
def generate_answer(question, context):
    prompt = f"""
You are a contract analysis assistant.
Answer only using the provided context.
Do not make up information.
If the answer cannot be found in the context, reply exactly:
Answer not found in the provided document context.
Context:
{context}
Question:
{question}
"""
    response = model.generate_content(prompt)
    return response.text

In [137]:
question = "Is there a penalty for late payment and what is the rate?"

results = retrieve_chunks(
    question,
    collection,
    embedding_model
)

context = build_context(results)

answer = generate_answer(
    question,
    context
)

print(answer)

Yes, if any invoiced amount is not received by SFDC by the due date, charges may accrue late interest. The rate is 1.5% of the outstanding balance per month, or the maximum rate permitted by law, whichever is lower.


In [138]:
import json
with open("ground_truth.json", "r") as f:
    ground_truth = json.load(f)
print("Questions Loaded:", len(ground_truth))

Questions Loaded: 10


In [246]:
import pandas as pd

eval_df = pd.read_json("evaluation_results.json")

eval_df = eval_df.rename(
    columns={
        "category": "Category",
        "judgement": "Judgement",
        "reason": "Reason"
    }
)

eval_df = eval_df[
    ["Category", "Judgement", "Reason"]
]

print(eval_df.head())

                    Category Judgement  \
0              Payment Terms     Match   
1       Late Payment Penalty     Match   
2  Delivery Deadline Penalty     Match   
3     Termination Conditions  No Match   
4  Termination Notice Period     Match   

                                              Reason  
0  The system answer conveys the exact same payme...  
1  The system answer accurately captures the spec...  
2  Both answers indicate that the information reg...  
3  The system answer lists entirely different ter...  
4  The system correctly identifies the 30-day wri...  


In [248]:
for item in evaluation_results[:3]:

    print("\nCATEGORY:", item["category"])
    print("\nEXPECTED:", item["expected_answer"])
    print("\nSYSTEM:", item["system_answer"])
    print("\n" + "="*100)


CATEGORY: Payment Terms

EXPECTED: Invoiced fees are due net 30 days from the invoice date unless otherwise stated in the Order Form.

SYSTEM: Unless otherwise stated in the Order Form, invoiced fees are due net 30 days from the invoice date.


CATEGORY: Late Payment Penalty

EXPECTED: Overdue charges may accrue interest at 1.5% of the outstanding balance per month or the maximum rate permitted by law, whichever is lower.

SYSTEM: Yes, if an invoiced amount is not received by the due date, charges may accrue late interest. The rate is 1.5% of the outstanding balance per month, or the maximum rate permitted by law, whichever is lower. (Note: The provided text also references a specific maximum rate of Legislative Decree no. 231/2002 in one section).


CATEGORY: Delivery Deadline Penalty

EXPECTED: No specific delivery milestone penalty is defined in the Salesforce MSA.

SYSTEM: Answer not found in the provided document context.



In [249]:
def judge_answer(expected_answer, system_answer):
    judge_prompt = f"""
You are evaluating a RAG system's answer against a ground truth answer extracted from a contract document.
Compare the two answers and classify the result as exactly one of:
Match
Partial Match
No Match
Then provide a single sentence explaining your classification.
Do not add any other commentary.
Ground Truth Answer: {expected_answer}
System Answer: {system_answer}
"""
    response = model.generate_content(judge_prompt)
    return response.text

In [250]:
judged_results = []

for item in evaluation_results:

    judge_output = judge_answer(
        item["expected_answer"],
        item["system_answer"]
    ).strip()

    if judge_output.startswith("Partial Match"):
        judgement = "Partial Match"
        reason = judge_output[len("Partial Match"):].strip(" .:-")

    elif judge_output.startswith("No Match"):
        judgement = "No Match"
        reason = judge_output[len("No Match"):].strip(" .:-")

    elif judge_output.startswith("Match"):
        judgement = "Match"
        reason = judge_output[len("Match"):].strip(" .:-")

    else:
        judgement = judge_output
        reason = ""
    reason = reason.strip()
    judged_results.append({
        "category": item["category"],
        "question": item["question"],
        "judgement": judgement,
        "reason": reason
    })

print("Judged Questions:", len(judged_results))

Judged Questions: 10


In [251]:
import json

with open("evaluation_results.json", "w") as f:
    json.dump(
        judged_results,
        f,
        indent=4
    )

In [252]:
for item in judged_results:
    print("\nCATEGORY:", item["category"])
    print(item["judgement"])
    print("-" * 80)


CATEGORY: Payment Terms
Match
--------------------------------------------------------------------------------

CATEGORY: Late Payment Penalty
Match
--------------------------------------------------------------------------------

CATEGORY: Delivery Deadline Penalty
Match
--------------------------------------------------------------------------------

CATEGORY: Termination Conditions
No Match
--------------------------------------------------------------------------------

CATEGORY: Termination Notice Period
Match
--------------------------------------------------------------------------------

CATEGORY: Limitation of Liability
Partial Match
--------------------------------------------------------------------------------

CATEGORY: Intellectual Property
No Match
--------------------------------------------------------------------------------

CATEGORY: Confidentiality Duration
No Match
--------------------------------------------------------------------------------

CATEGORY: Governi

In [253]:
import pandas as pd

df = pd.DataFrame(judged_results)

df

,category,question,judgement,reason
0,Payment Terms,When is an invoice due after it is issued?,Match,The system answer conveys the exact same payme...
1,Late Payment Penalty,Is there a penalty for late payment and what i...,Match,The system answer correctly identifies the int...
2,Delivery Deadline Penalty,What is the penalty for the provider missing a...,Match,Both the ground truth and the system answer in...
3,Termination Conditions,Under what conditions can either party termina...,No Match,The system answer describes termination condit...
4,Termination Notice Period,How many days written notice is required to te...,Match,The system answer correctly identifies the 30-...
5,Limitation of Liability,What is the maximum liability cap defined in t...,Partial Match,While the system answer accurately reflects th...
6,Intellectual Property,Who retains ownership of intellectual property...,No Match,The system failed to retrieve or identify the ...
7,Confidentiality Duration,How long do confidentiality obligations remain...,No Match,The system failed to retrieve the information ...
8,Governing Law,Which state or jurisdiction governs this agree...,Partial Match,The system answer provides accurate informatio...
9,Dispute Resolution,What is the defined process for resolving disp...,Match,The system answer includes all the essential c...


In [254]:
match_count = sum(
    "Match" in j["judgement"] and
    "Partial" not in j["judgement"]
    for j in judged_results
)
partial_count = sum(
    "Partial Match" in j["judgement"]
    for j in judged_results
)
no_match_count = sum(
    "No Match" in j["judgement"]
    for j in judged_results
)
print("Match:", match_count)
print("Partial:", partial_count)
print("No Match:", no_match_count)

Match: 8
Partial: 2
No Match: 3


In [255]:
import json

with open("evaluation_results.json", "r") as f:
    data = json.load(f)

print(data[1])

{'category': 'Late Payment Penalty', 'question': 'Is there a penalty for late payment and what is the rate?', 'judgement': 'Match', 'reason': 'The system answer correctly identifies the interest rate and the conditions for accrual, providing the same essential information as the ground truth despite the additional note'}
